# 步骤 13：在 Colab 上训练零件检测模型（YOLO26n）

**开始前：**
1. `parts_data.zip`（里面直接是 `images/`、`labels/` 两个文件夹）已经上传到 Google Drive 的 `MyDrive/yolohtml/`。
2. 菜单 **执行阶段（Runtime）→ 变更执行阶段类型（Change runtime type）→ 硬件加速器选 T4 GPU** → 储存。

**用法：** 从上到下，一个单元格一个单元格地点左边的 ▶ 运行，等它跑完（左边不再转圈）再运行下一个。
某个单元格报错就停下，把红色的输出复制下来。

训练结果直接存到 Google Drive 的 `MyDrive/yolohtml/runs/parts/`，Colab 断线也不会丢。

## 1. 确认 GPU、连接 Google Drive

会弹出授权窗口，选你的 Google 账号并允许。最后一行应该显示 `True` 和显卡名（如 Tesla T4）。

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import torch
print('GPU 可用：', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '（没有 GPU！请到"执行阶段"里改成 T4 GPU）')

## 2. 下载代码、安装依赖

大约 1–2 分钟。Colab 自带 GPU 版 torch，这里不重装 torch。

In [ ]:
%cd /content
!rm -rf /content/yolohtml
!git clone -q --depth 1 -b claude/upbeat-johnson-d955qn https://github.com/shixin08052/yolohtml.git /content/yolohtml
%cd /content/yolohtml/training
!pip install -q -r requirements.txt
import ultralytics; print('ultralytics', ultralytics.__version__)

## 3. 解压数据并划分训练 / 验证 / 测试集（8 : 1 : 1）

记下打印出来的各集合照片数和框数（写报告用）。

In [ ]:
DRIVE = '/content/drive/MyDrive/yolohtml'
!rm -rf data && mkdir -p data && unzip -q -o {DRIVE}/parts_data.zip -d data
import os
n_img = len([f for f in os.listdir('data/images') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if os.path.isdir('data/images') else 0
n_lab = len([f for f in os.listdir('data/labels') if f.endswith('.txt') and f != 'labels.txt']) if os.path.isdir('data/labels') else 0
print('照片', n_img, '张，标注', n_lab, '个')
assert n_img > 0, '没找到 data/images：压缩包第一层应该直接是 images 和 labels 两个文件夹'
!python split_dataset.py --images data/images --labels data/labels

## 4. 训练

T4 上 100 张照片大约 30–60 分钟。会不停打印每一轮的结果，最后一行是 `训练完成`。

- 显存不够（`CUDA out of memory`）：把命令末尾加上 ` --batch 8`，还不行改 ` --batch 4`。
- **中途断线 / 超时：** 重新连接后，先运行第 1、2、3 个单元格，然后**跳过这个单元格**，运行下面的"4b 接着训练"。

In [ ]:
!python train.py --project {DRIVE}/runs --name parts

### 4b. 接着训练（只在断线后用，正常情况跳过）

In [ ]:
# !python train.py --project {DRIVE}/runs --name parts --resume

## 5. 查看训练曲线和混淆矩阵

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png']:
    p = f'{DRIVE}/runs/parts/{f}'
    if os.path.exists(p):
        print(f); display(Image(p, width=900))

## 6. 在测试集上评估

看 `all`、`resistor`、`led` 三行的 mAP50，期望 ≥ 0.9。把整段输出复制下来（写报告用）。

In [ ]:
!yolo detect val model={DRIVE}/runs/parts/weights/best.pt data=datasets/parts/parts.yaml split=test imgsz=1024

## 7. 导出 ONNX（网页用）

期望打印输入 `[1, 3, 1024, 1024]`、输出 `[1, 300, 6]`、"端到端输出"。

In [ ]:
!python export.py --weights {DRIVE}/runs/parts/weights/best.pt
!python export.py --weights yolo26n.pt --copy-to ../public/models/yolo26n_coco.onnx
!ls -la ../public/models

## 8. 生成对比数据（步骤 14 核对网页推理用）

In [ ]:
# 用 CPU 跑 ONNX（和网页的 WASM 一样）。GPU 版 onnxruntime 与 Colab 的 CUDA 版本常对不上，会报错
!pip uninstall -q -y onnxruntime-gpu
!pip install -q --force-reinstall --no-deps onnxruntime
!python predict_dump.py --weights ../public/models/best.onnx datasets/parts/images/test --out dumps/best_onnx --device cpu
import glob, shutil
shutil.rmtree('../public/samples/real', ignore_errors=True)  # 清掉仓库里上一版的对比照片
os.makedirs('../public/samples/real')
picked = sorted(glob.glob('datasets/parts/images/test/*.jpg'))[:3]
for p in picked:
    stem = os.path.splitext(os.path.basename(p))[0]
    shutil.copy(p, '../public/samples/real/')
    shutil.copy(f'dumps/best_onnx/{stem}.detections.json', '../public/samples/real/')
print('选了这 3 张：', picked)

## 9. 打包结果到 Google Drive

运行完后，到 Google Drive 的 `MyDrive/yolohtml/` 下载 `step13_outputs.zip`，以及 `results.png`、`confusion_matrix.png`。

In [ ]:
!cd .. && rm -f {DRIVE}/step13_outputs.zip && zip -q -r {DRIVE}/step13_outputs.zip public/models public/samples/real
!cp {DRIVE}/runs/parts/results.png {DRIVE}/runs/parts/confusion_matrix.png {DRIVE}/
!ls -la {DRIVE}
print('完成！')

## 10. 独立测试集评估（可选）

独立测试集：另外拍的、**从未参与训练**的照片，标注好后打包成 `holdout_data.zip`（里面直接是 `images/`、`labels/`），上传到 Drive 的 `MyDrive/yolohtml/`。

新开的会话先运行单元格 1、2（不用跑 3–9，但第 4 步训练出的 `best.pt` 要已经在 Drive 上）。这里同时评估第二版模型（`best.pt`）和第一版模型（只见过红色 LED，从 GitHub 下载），对比表直接贴进报告。

In [ ]:
DRIVE = '/content/drive/MyDrive/yolohtml'
!rm -rf holdout && mkdir -p holdout && unzip -q -o {DRIVE}/holdout_data.zip -d holdout
# ONNX 用 CPU 版 onnxruntime
!pip uninstall -q -y onnxruntime-gpu
!pip install -q --force-reinstall --no-deps onnxruntime
# 第一版模型（只见过红色 LED）
!wget -q -O best_v1.onnx https://raw.githubusercontent.com/shixin08052/yolohtml/094e3d0/public/models/best.onnx
!python eval_holdout.py --data holdout --models v2={DRIVE}/runs/parts/weights/best.pt v1=best_v1.onnx --project {DRIVE}/runs/holdout --name eval